# FT-Transformer — Employee Resignation Prediction

## Member 1 Model

**Model:** FT-Transformer  
**Task:** Binary classification of employee resignation risk  
**Target:** `Resigned`  
**Positive class:** `True` / 1

## Purpose

This notebook trains and evaluates an **FT-Transformer** model for employee resignation prediction.

FT-Transformer is a neural-network architecture designed for tabular data. It can learn from both numerical and categorical features while using Transformer-based attention to model relationships between features.

### Main objectives

- Use only the **80,000-row development partition** for model development and evaluation.
- Keep the **20,000-row formal holdout** excluded from model training, preprocessing decisions, tuning, and model selection.
- Use all **17 approved predictor features**:
  - 13 numerical/ordinal predictors
  - 4 categorical predictors
- Apply FT-Transformer-specific preprocessing.
- Fit numerical preprocessing and categorical vocabularies only on the training portion of each fold.
- Reuse the existing fixed **5-fold cross-validation** assignments.
- Generate one out-of-fold prediction for every development employee.
- Evaluate the model primarily using **PR-AUC**.
- Also report ROC-AUC, Brier Score, balanced accuracy, precision, recall, and F1 where appropriate.
- Save reproducible model results under `results/models/ft_transformer/`.

## Evaluation principle

All preprocessing and model fitting must be performed independently inside each training fold.

The formal holdout set is reserved for the final locked evaluation stage and is not used in this notebook.

## FT-Transformer Methodology and Leakage Rules

To keep the evaluation fair and prevent data leakage, this notebook follows the same frozen development split used throughout the project.

### Data usage

- **Development partition:** 80,000 employees
- **Formal holdout:** 20,000 employees
- The formal holdout is **not used** in this notebook.
- The existing 5 development folds are reused exactly as created in Notebook 02.
- Each fold contains 16,000 validation rows.

For each cross-validation round:

- **Training:** 64,000 employees
- **Validation:** 16,000 employees

### Features

FT-Transformer will use all **17 approved predictors**.

#### Numerical / ordinal features

1. `Age`
2. `Years_At_Company`
3. `Performance_Score`
4. `Monthly_Salary`
5. `Work_Hours_Per_Week`
6. `Projects_Handled`
7. `Overtime_Hours`
8. `Sick_Days`
9. `Remote_Work_Frequency`
10. `Team_Size`
11. `Training_Hours`
12. `Promotions`
13. `Employee_Satisfaction_Score`

#### Categorical features

1. `Department`
2. `Gender`
3. `Job_Title`
4. `Education_Level`

### Leakage prevention

Preprocessing must be learned only from the training portion of each fold.

For every fold:

1. Select the 64,000 training rows.
2. Select the 16,000 validation rows.
3. Fit numerical preprocessing using only the training rows.
4. Build categorical mappings using only the training rows.
5. Transform the validation rows using those already-fitted mappings.
6. Train FT-Transformer only on the training rows.
7. Predict probabilities for the validation rows.
8. Store those predictions as out-of-fold predictions.

The validation fold must never influence preprocessing or model fitting.

### Evaluation

The primary metric is **PR-AUC** because resignation is an imbalanced target with approximately 10% positive cases.

Supporting metrics include:

- ROC-AUC
- Brier Score
- Balanced Accuracy
- Precision
- Recall
- F1 Score

The final 20,000-row holdout remains untouched until the locked final evaluation stage.

In [3]:
# -----------------------------
# Cell 3: Imports and project setup
# -----------------------------

import sys
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import torch


# Resolve the repository root.
#
# This notebook is inside:
# notebooks/05_member_1/
#
# So ../.. normally points to the project root.
PROJECT_ROOT = Path("../..").resolve()


# Fallback:
# If the notebook is launched from a different working directory,
# search nearby folders for the feature contract.
if not (PROJECT_ROOT / "configs" / "feature_contract.yaml").exists():

    for candidate in [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]:

        if (candidate / "configs" / "feature_contract.yaml").exists():
            PROJECT_ROOT = candidate.resolve()
            break


# Add the repository root to Python's import path
# so we can import modules from src/.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Project-specific FT-Transformer code
from src.models.ft_transformer import FTTransformerAdapter
from src.models.runner import run_5fold_cv


# Show all DataFrame columns when displaying tables.
pd.set_option("display.max_columns", None)


# Important project paths
CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "models"
    / "ft_transformer.yaml"
)

FEATURE_CONTRACT_PATH = (
    PROJECT_ROOT
    / "configs"
    / "feature_contract.yaml"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "ft_transformer"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Project root:", PROJECT_ROOT)
print("FT-Transformer config:", CONFIG_PATH)
print("Feature contract:", FEATURE_CONTRACT_PATH)
print("Results directory:", RESULTS_DIR)

print("\nPyTorch version:", torch.__version__)
print(
    "CUDA available:",
    torch.cuda.is_available()
)

Project root: D:\Desktop\Employee_Resignation_Prediction
FT-Transformer config: D:\Desktop\Employee_Resignation_Prediction\configs\models\ft_transformer.yaml
Feature contract: D:\Desktop\Employee_Resignation_Prediction\configs\feature_contract.yaml
Results directory: D:\Desktop\Employee_Resignation_Prediction\results\models\ft_transformer

PyTorch version: 2.13.0+cpu
CUDA available: False


In [4]:
# -----------------------------
# Cell 4: Load model config and feature contract
# -----------------------------

# Safety checks:
# Make sure both required configuration files exist.
assert CONFIG_PATH.exists(), (
    f"FT-Transformer config not found: {CONFIG_PATH}"
)

assert FEATURE_CONTRACT_PATH.exists(), (
    f"Feature contract not found: {FEATURE_CONTRACT_PATH}"
)


# Load the FT-Transformer model configuration.
with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    ft_config = yaml.safe_load(file)


# Load the shared project feature contract.
with open(FEATURE_CONTRACT_PATH, "r", encoding="utf-8") as file:
    feature_contract = yaml.safe_load(file)


print("FT-Transformer configuration loaded successfully.")
print("Feature contract loaded successfully.")

print("\nFT-Transformer config:")
display(ft_config)

print("\nFeature contract top-level sections:")
print(list(feature_contract.keys()))

FT-Transformer configuration loaded successfully.
Feature contract loaded successfully.

FT-Transformer config:


{'model': {'name': 'FT-Transformer',
  'model_id': 'ft_transformer',
  'member': 'Member 1',
  'algorithm_family': 'Feature Tokenizer Transformer for Tabular Data',
  'implementation': 'rtdl_revisiting_models.FTTransformer'},
 'reproducibility': {'seed': 42, 'device': 'auto'},
 'fixed_params': {'seed': 42,
  'n_blocks': 3,
  'd_block': 64,
  'attention_n_heads': 4,
  'ffn_d_hidden': 128,
  'ffn_dropout': 0.1,
  'attention_dropout': 0.1,
  'residual_dropout': 0.0},
 'validation': {'cv_folds': 5,
  'batch_size': 512,
  'max_epochs': 100,
  'patience': 15,
  'lr_scheduler': 'ReduceLROnPlateau',
  'validation_fraction_inner': 0.15},
 'search_space': {'n_blocks': {'type': 'int',
   'low': 2,
   'high': 6,
   'step': 1,
   'default': 3},
  'd_block': {'type': 'int',
   'low': 32,
   'high': 128,
   'step': 32,
   'default': 64},
  'attention_n_heads': {'type': 'int',
   'low': 2,
   'high': 8,
   'step': 2,
   'default': 4},
  'ffn_d_hidden_multiplier': {'type': 'float',
   'low': 1.0,
   'h


Feature contract top-level sections:
['contract', 'dataset', 'split_registry', 'target', 'identifier', 'excluded_features', 'categorical_features', 'numerical_features', 'primary_model_features', 'feature_counts', 'sensitive_attributes', 'sensitive_attribute_policy', 'age_bands', 'data_quality_policy', 'preprocessing_policy', 'model_representations', 'optional_logistic_regression', 'cross_validation_contract', 'prediction_output_contract', 'change_control']


In [5]:
# -----------------------------
# Cell 5: Load dataset and frozen split registry
# -----------------------------

# Define the canonical project data paths.
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Extended_Employee_Performance_and_Productivity_Data.csv"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_registry.csv"
)


# Safety checks before loading.
assert DATA_PATH.exists(), (
    f"Dataset not found: {DATA_PATH}"
)

assert SPLIT_PATH.exists(), (
    f"Split registry not found: {SPLIT_PATH}"
)


# Load the raw dataset and the already-frozen split registry.
df = pd.read_csv(DATA_PATH)
split_registry = pd.read_csv(SPLIT_PATH)


print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

print("\nSplit registry loaded successfully.")
print("Split registry shape:", split_registry.shape)

print("\nDataset columns:")
print(df.columns.tolist())

print("\nSplit registry columns:")
print(split_registry.columns.tolist())

Dataset loaded successfully.
Dataset shape: (100000, 20)

Split registry loaded successfully.
Split registry shape: (100000, 3)

Dataset columns:
['Employee_ID', 'Department', 'Gender', 'Age', 'Job_Title', 'Hire_Date', 'Years_At_Company', 'Education_Level', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score', 'Resigned']

Split registry columns:
['Employee_ID', 'partition', 'cv_fold']


In [8]:
# -----------------------------
# Cell 6: Merge split registry and isolate development data
# -----------------------------

# Make sure Employee_ID is unique in both files before merging.
assert df["Employee_ID"].is_unique, (
    "Employee_ID is not unique in the raw dataset."
)

assert split_registry["Employee_ID"].is_unique, (
    "Employee_ID is not unique in the split registry."
)


# Merge the frozen partition/fold information onto the raw dataset.
data = df.merge(
    split_registry,
    on="Employee_ID",
    how="left",
    validate="one_to_one"
)


# Every employee must have a split assignment.
assert data["partition"].notna().all(), (
    "Some employees do not have a partition assignment."
)


print("Merged dataset shape:", data.shape)

print("\nPartition counts:")
print(data["partition"].value_counts(dropna=False))

print("\nCV fold counts:")
print(data["cv_fold"].value_counts(dropna=False).sort_index())


# Keep ONLY the development partition.
development_df = (
    data.loc[data["partition"] == "development"]
    .copy()
    .reset_index(drop=True)
)


# Development rows all have valid fold assignments,
# so convert cv_fold from float to integer for cleaner use later.
development_df["cv_fold"] = (
    development_df["cv_fold"]
    .astype(int)
)


print("\nDevelopment dataset shape:", development_df.shape)

print(
    "Formal holdout rows excluded:",
    int((data["partition"] != "development").sum())
)

print("\nDevelopment fold counts:")
print(
    development_df["cv_fold"]
    .value_counts()
    .sort_index()
)


# Safety checks required by the project design.
assert len(development_df) == 80_000, (
    f"Expected 80,000 development rows, found {len(development_df):,}."
)

assert development_df["cv_fold"].notna().all(), (
    "Development data contains missing CV fold assignments."
)

assert set(development_df["cv_fold"].unique()) == {1, 2, 3, 4, 5}, (
    "Expected development CV folds 1, 2, 3, 4, and 5."
)

assert (
    development_df["cv_fold"]
    .value_counts()
    .sort_index()
    .eq(16_000)
    .all()
), "Each development fold should contain exactly 16,000 rows."


print("\nDevelopment partition isolation checks passed.")

Merged dataset shape: (100000, 22)

Partition counts:
partition
development    80000
holdout        20000
Name: count, dtype: int64

CV fold counts:
cv_fold
1.0    16000
2.0    16000
3.0    16000
4.0    16000
5.0    16000
NaN    20000
Name: count, dtype: int64

Development dataset shape: (80000, 22)
Formal holdout rows excluded: 20000

Development fold counts:
cv_fold
1    16000
2    16000
3    16000
4    16000
5    16000
Name: count, dtype: int64

Development partition isolation checks passed.


In [9]:
# -----------------------------
# Cell 7: Inspect the approved feature contract
# -----------------------------

# Read the feature groups defined by the project contract.
numerical_features = feature_contract["numerical_features"]
categorical_features = feature_contract["categorical_features"]
primary_model_features = feature_contract["primary_model_features"]

target_config = feature_contract["target"]
excluded_features = feature_contract["excluded_features"]


print("Numerical / ordinal features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nPrimary model features:")
print(primary_model_features)

print("\nTarget configuration:")
print(target_config)

print("\nExcluded features:")
print(excluded_features)

print("\nPython object types:")
print("numerical_features:", type(numerical_features).__name__)
print("categorical_features:", type(categorical_features).__name__)
print("primary_model_features:", type(primary_model_features).__name__)
print("target:", type(target_config).__name__)
print("excluded_features:", type(excluded_features).__name__)

Numerical / ordinal features:
[{'name': 'Age', 'semantic_type': 'integer_numeric', 'sensitive_attribute': True}, {'name': 'Years_At_Company', 'semantic_type': 'integer_ordinal', 'sensitive_attribute': False}, {'name': 'Performance_Score', 'semantic_type': 'integer_ordinal', 'sensitive_attribute': False}, {'name': 'Monthly_Salary', 'semantic_type': 'numeric', 'sensitive_attribute': False, 'note': 'Salary unit/currency is unverified. Monthly_Salary is also deterministically related to Job_Title and Performance_Score in the audited dataset. Retain it in the primary feature set; salary-removal sensitivity analysis is performed later.\n'}, {'name': 'Work_Hours_Per_Week', 'semantic_type': 'integer_numeric', 'sensitive_attribute': False}, {'name': 'Projects_Handled', 'semantic_type': 'integer_count', 'sensitive_attribute': False}, {'name': 'Overtime_Hours', 'semantic_type': 'integer_numeric', 'sensitive_attribute': False}, {'name': 'Sick_Days', 'semantic_type': 'integer_count', 'sensitive_att

In [10]:
# -----------------------------
# Cell 8: Build predictors and target
# -----------------------------

# Extract column names from the feature-contract metadata.
numerical_feature_names = [
    item["name"]
    for item in numerical_features
]

categorical_feature_names = [
    item["name"]
    for item in categorical_features
]

target_column = target_config["column"]


# --------------------------------------------------
# Validate the feature contract before modeling
# --------------------------------------------------

assert len(numerical_feature_names) == 13, (
    f"Expected 13 numerical/ordinal features, "
    f"found {len(numerical_feature_names)}."
)

assert len(categorical_feature_names) == 4, (
    f"Expected 4 categorical features, "
    f"found {len(categorical_feature_names)}."
)

assert len(primary_model_features) == 17, (
    f"Expected 17 primary model features, "
    f"found {len(primary_model_features)}."
)


# The two feature groups together must match
# the approved 17-feature primary model contract.
assert set(
    numerical_feature_names + categorical_feature_names
) == set(primary_model_features), (
    "Numerical + categorical features do not match "
    "the primary model feature contract."
)


# Make sure every required column exists in the development data.
required_columns = (
    primary_model_features
    + [target_column, "Employee_ID", "cv_fold"]
)

missing_columns = [
    column
    for column in required_columns
    if column not in development_df.columns
]

assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)


# --------------------------------------------------
# Create model inputs
# --------------------------------------------------

# X contains ONLY the 17 approved predictors.
X = development_df[
    primary_model_features
].copy()


# Convert the Boolean resignation target to 0/1.
# False -> 0
# True  -> 1
y = development_df[
    target_column
].astype(int).copy()


# Keep identifiers and frozen fold labels separately.
employee_ids = development_df[
    "Employee_ID"
].copy()

cv_folds = development_df[
    "cv_fold"
].copy()


# --------------------------------------------------
# Safety checks
# --------------------------------------------------

assert X.shape == (80_000, 17), (
    f"Unexpected predictor shape: {X.shape}"
)

assert len(y) == 80_000
assert len(employee_ids) == 80_000
assert len(cv_folds) == 80_000

assert set(y.unique()).issubset({0, 1}), (
    "Target must contain only binary values 0 and 1."
)

assert "Employee_ID" not in X.columns
assert "Hire_Date" not in X.columns
assert target_column not in X.columns


# --------------------------------------------------
# Summary
# --------------------------------------------------

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nNumerical / ordinal features:", len(numerical_feature_names))
print("Categorical features:", len(categorical_feature_names))
print("Total predictors:", X.shape[1])

print("\nTarget counts:")
print(y.value_counts().sort_index())

print("\nTarget percentages:")
print(
    y.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(3)
)

print("\nMissing values in predictor matrix:", int(X.isna().sum().sum()))

print("\nFeature and target validation checks passed.")

Predictor matrix shape: (80000, 17)
Target shape: (80000,)

Numerical / ordinal features: 13
Categorical features: 4
Total predictors: 17

Target counts:
Resigned
0    71992
1     8008
Name: count, dtype: int64

Target percentages:
Resigned
0    89.99
1    10.01
Name: proportion, dtype: float64

Missing values in predictor matrix: 0

Feature and target validation checks passed.


In [11]:
# -----------------------------
# Cell 9: Verify target balance across frozen CV folds
# -----------------------------

# Create a compact summary for each of the five development folds.
fold_summary = (
    development_df
    .groupby("cv_fold")[target_column]
    .agg(
        total_rows="size",
        resigned_count="sum",
        resignation_rate="mean"
    )
    .reset_index()
)

# Number of employees who did not resign.
fold_summary["not_resigned_count"] = (
    fold_summary["total_rows"]
    - fold_summary["resigned_count"]
)

# Convert the resignation rate to percentage form.
fold_summary["resignation_rate_pct"] = (
    fold_summary["resignation_rate"] * 100
).round(3)


# Reorder columns for easier reading.
fold_summary = fold_summary[
    [
        "cv_fold",
        "total_rows",
        "not_resigned_count",
        "resigned_count",
        "resignation_rate_pct"
    ]
]


display(fold_summary)


# --------------------------------------------------
# Validation checks
# --------------------------------------------------

assert len(fold_summary) == 5, (
    "Expected exactly 5 development folds."
)

assert (fold_summary["total_rows"] == 16_000).all(), (
    "Every validation fold should contain exactly 16,000 employees."
)

assert (fold_summary["resigned_count"] > 0).all(), (
    "Every fold must contain positive resignation cases."
)

assert (fold_summary["not_resigned_count"] > 0).all(), (
    "Every fold must contain negative resignation cases."
)


print(
    "\nOverall development resignation rate:",
    f"{y.mean() * 100:.3f}%"
)

print("\nFrozen fold balance checks passed.")

,cv_fold,total_rows,not_resigned_count,resigned_count,resignation_rate_pct
0,1,16000,14399,1601,10.006
1,2,16000,14399,1601,10.006
2,3,16000,14398,1602,10.013
3,4,16000,14398,1602,10.013
4,5,16000,14398,1602,10.013



Overall development resignation rate: 10.010%

Frozen fold balance checks passed.


In [12]:
# -----------------------------
# Cell 10: Inspect FT-Transformer and CV runner APIs
# -----------------------------

import inspect


print("FTTransformerAdapter constructor:")
print(
    inspect.signature(
        FTTransformerAdapter
    )
)


print("\nFTTransformerAdapter.fit:")
print(
    inspect.signature(
        FTTransformerAdapter.fit
    )
)


print("\nFTTransformerAdapter.predict_proba:")
print(
    inspect.signature(
        FTTransformerAdapter.predict_proba
    )
)


print("\nrun_5fold_cv:")
print(
    inspect.signature(
        run_5fold_cv
    )
)


print("\nFTTransformerAdapter documentation:")
print(
    inspect.getdoc(
        FTTransformerAdapter
    )
)


FTTransformerAdapter constructor:
(name: 'str' = 'FT-Transformer', model_id: 'str' = 'ft_transformer', params: 'dict[str, Any] | None' = None, categorical_features: 'list[str] | None' = None) -> 'None'

FTTransformerAdapter.fit:
(self, X: 'pd.DataFrame', y: 'pd.Series | np.ndarray', epochs: 'int' = 50, batch_size: 'int' = 512) -> 'FTTransformerAdapter'

FTTransformerAdapter.predict_proba:
(self, X: 'pd.DataFrame', batch_size: 'int' = 1024) -> 'np.ndarray'

run_5fold_cv:
(adapter_factory: 'Callable[[], ModelAdapter]', data: 'pd.DataFrame', split_registry: 'pd.DataFrame', feature_cols: 'Sequence[str]', target_col: 'str' = 'Resigned', id_col: 'str' = 'Employee_ID', fold_col: 'str' = 'cv_fold', partition_col: 'str' = 'partition', development_label: 'str' = 'development', holdout_label: 'str' = 'holdout', expected_folds: 'Sequence[int]' = (1, 2, 3, 4, 5), expected_model_name: 'str | None' = None) -> 'OOFResult'

FTTransformerAdapter documentation:
FT-Transformer model adapter with fold-loca

In [13]:
# -----------------------------
# Cell 11: Construct the FT-Transformer adapter
# -----------------------------

# Read the fixed model settings from the YAML configuration.
model_name = ft_config["model"]["name"]
model_id = ft_config["model"]["model_id"]

fixed_params = ft_config["fixed_params"].copy()


print("Model name:", model_name)
print("Model ID:", model_id)

print("\nFixed parameters:")
for key, value in fixed_params.items():
    print(f"  {key}: {value}")


# --------------------------------------------------
# Create an adapter instance
# --------------------------------------------------
#
# This does NOT train the neural network.
#
# It only creates the Python object that will later
# perform fold-local preprocessing and model fitting.

ft_adapter_check = FTTransformerAdapter(
    name=model_name,
    model_id=model_id,
    params=fixed_params,
    categorical_features=categorical_feature_names
)


# --------------------------------------------------
# Basic adapter validation
# --------------------------------------------------

assert ft_adapter_check.name == "FT-Transformer", (
    f"Unexpected model name: {ft_adapter_check.name}"
)

assert ft_adapter_check.model_id == "ft_transformer", (
    f"Unexpected model ID: {ft_adapter_check.model_id}"
)


print("\nAdapter created successfully.")
print("Adapter class:", type(ft_adapter_check).__name__)


# Show the adapter metadata before training.
adapter_metadata = ft_adapter_check.get_metadata()

print("\nAdapter metadata:")
display(adapter_metadata)

Model name: FT-Transformer
Model ID: ft_transformer

Fixed parameters:
  seed: 42
  n_blocks: 3
  d_block: 64
  attention_n_heads: 4
  ffn_d_hidden: 128
  ffn_dropout: 0.1
  attention_dropout: 0.1
  residual_dropout: 0.0

Adapter created successfully.
Adapter class: FTTransformerAdapter

Adapter metadata:


{'name': 'FT-Transformer',
 'model_id': 'ft_transformer',
 'fitted': False,
 'params': {'seed': 42,
  'n_blocks': 3,
  'd_block': 64,
  'attention_n_heads': 4,
  'ffn_d_hidden': 128,
  'ffn_dropout': 0.1,
  'attention_dropout': 0.1,
  'residual_dropout': 0.0},
 'categorical_features': ['Department',
  'Gender',
  'Job_Title',
  'Education_Level'],
 'feature_names': []}

In [14]:
# -----------------------------
# Cell 12: Inspect CV runner training behaviour
# -----------------------------

import inspect


runner_source = inspect.getsource(run_5fold_cv)


# Print only the important lines from the runner rather than
# dumping the entire source file.
important_terms = [
    "adapter_factory",
    ".fit(",
    "predict_proba",
    "OOFResult",
    "fold",
]

print("Relevant run_5fold_cv() source lines:\n")

for line_number, line in enumerate(
    runner_source.splitlines(),
    start=1
):
    if any(term in line for term in important_terms):
        print(
            f"{line_number:>3}: {line}"
        )


print("\nFTTransformerAdapter.fit defaults:")

fit_signature = inspect.signature(
    FTTransformerAdapter.fit
)

print(
    "epochs default:",
    fit_signature.parameters["epochs"].default
)

print(
    "batch_size default:",
    fit_signature.parameters["batch_size"].default
)


print("\nYAML validation settings:")

print(
    "max_epochs:",
    ft_config["validation"]["max_epochs"]
)

print(
    "batch_size:",
    ft_config["validation"]["batch_size"]
)

print(
    "patience:",
    ft_config["validation"]["patience"]
)

print(
    "inner validation fraction:",
    ft_config["validation"]["validation_fraction_inner"]
)

Relevant run_5fold_cv() source lines:

  1: def run_5fold_cv(
  2:     adapter_factory: Callable[[], ModelAdapter],
  8:     fold_col: str = "cv_fold",
 12:     expected_folds: Sequence[int] = (1, 2, 3, 4, 5),
 14: ) -> OOFResult:
 15:     """Execute leakage-safe 5-fold cross-validation across the development partition.
 17:     This function coordinates out-of-fold evaluation across the 5 frozen
 18:     cross-validation folds defined in the split registry. It enforces strict
 20:     every fold, validates probability outputs, and produces canonical OOF artifacts.
 24:     adapter_factory : Callable[[], ModelAdapter]
 26:         `ModelAdapter` object. A fresh instance is constructed for each fold
 27:         to prevent cross-fold state or preprocessor leakage.
 32:         Standardized split registry defining `id_col`, `partition_col`, and `fold_col`.
 39:     fold_col : str, default="cv_fold"
 40:         Name of the cross-validation fold column.
 47:     expected_folds : Sequence[

In [15]:
# Show the exact area where run_5fold_cv() calls adapter.fit()

runner_lines = inspect.getsource(run_5fold_cv).splitlines()

for i, line in enumerate(runner_lines):
    if ".fit(" in line:
        start = max(0, i - 5)
        end = min(len(runner_lines), i + 8)

        for j in range(start, end):
            print(f"{j + 1:>3}: {runner_lines[j]}")

 65: 
 66:     Notes
 67:     -----
 68:     **Inner Validation Boundary:** The runner fits each fold adapter using strictly
 69:     the 4 outer-training folds (`train_df[feature_cols]`, `train_df[target_col]`).
 70:     The outer validation fold is NEVER passed to `.fit()` as an evaluation or
 71:     early-stopping set. Adapters requiring validation during training (e.g. for early
 72:     stopping) must partition their own training fold internally.
 73:     """
 74:     # ---------------------------------------------------------
 75:     # 1. Input and Schema Validation
 76:     # ---------------------------------------------------------
 77:     required_registry_cols = {id_col, partition_col, fold_col}
208:         X_val = val_df[list(feature_cols)]
209:         y_val = val_df[target_col]
210: 
211:         # Fit adapter strictly on outer training data
212:         t0_fit = time.perf_counter()
213:         adapter.fit(X=X_train, y=y_train)
214:         fit_duration = time.perf_co

In [16]:
# -----------------------------
# Cell 13: Inspect FT-Transformer fit() behaviour
# -----------------------------

fit_source = inspect.getsource(
    FTTransformerAdapter.fit
)

fit_lines = fit_source.splitlines()


# Print the complete fit() method with line numbers.
print("FTTransformerAdapter.fit() source:\n")

for line_number, line in enumerate(
    fit_lines,
    start=1
):
    print(
        f"{line_number:>3}: {line}"
    )

FTTransformerAdapter.fit() source:

  1:     def fit(
  2:         self,
  3:         X: pd.DataFrame,
  4:         y: pd.Series | np.ndarray,
  5:         epochs: int = 50,
  6:         batch_size: int = 512,
  7:     ) -> FTTransformerAdapter:
  8:         """Fit fold-local preprocessors and FT-Transformer network on training fold data."""
  9:         try:
 10:             import torch
 11:             import torch.nn as nn
 12:             from rtdl_revisiting_models import FTTransformer
 13:             from torch.utils.data import DataLoader, TensorDataset
 14:         except ImportError as exc:
 15:             raise ImportError(
 16:                 "torch and rtdl-revisiting-models are required for FTTransformerAdapter."
 17:             ) from exc
 18: 
 19:         self._feature_names = list(X.columns)
 20:         x_num, x_cat = self._preprocess(X, fit=True)
 21:         y_arr = np.asarray(y, dtype=np.float32)
 22: 
 23:         device = torch.device("cuda" if torch.cuda.is

In [17]:
# Show only the important FT-Transformer training lines

important_terms = [
    "train_test_split",
    "validation",
    "val_",
    "patience",
    "scheduler",
    "ReduceLROnPlateau",
    "optimizer",
    "Adam",
    "AdamW",
    "learning_rate",
    "lr",
    "weight_decay",
    "epochs",
    "BCE",
    "pos_weight",
    "CrossEntropy",
    "DataLoader",
    "shuffle",
]

for line_number, line in enumerate(
    fit_lines,
    start=1
):
    if any(
        term.lower() in line.lower()
        for term in important_terms
    ):
        print(f"{line_number:>3}: {line}")

  5:         epochs: int = 50,
 13:             from torch.utils.data import DataLoader, TensorDataset
 52:         loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
 54:         optimizer = torch.optim.AdamW(
 56:             lr=self.params.get("learning_rate", 0.001),
 57:             weight_decay=self.params.get("weight_decay", 1e-5),
 59:         criterion = nn.BCEWithLogitsLoss()
 62:         for _ in range(epochs):
 64:                 optimizer.zero_grad()
 77:                 optimizer.step()


In [18]:
# -----------------------------
# Cell 14: Inspect FT-Transformer preprocessing
# -----------------------------

preprocess_source = inspect.getsource(
    FTTransformerAdapter._preprocess
)

preprocess_lines = preprocess_source.splitlines()


print("FTTransformerAdapter._preprocess() source:\n")

for line_number, line in enumerate(
    preprocess_lines,
    start=1
):
    print(
        f"{line_number:>3}: {line}"
    )

FTTransformerAdapter._preprocess() source:

  1:     def _preprocess(
  2:         self,
  3:         X: pd.DataFrame,
  4:         fit: bool = False,
  5:     ) -> tuple[np.ndarray | None, np.ndarray | None]:
  6:         """Preprocess inputs into scaled continuous matrix and categorical integer matrix."""
  7:         from sklearn.preprocessing import StandardScaler
  8: 
  9:         X_proc = X.copy()
 10:         self._num_features = [
 11:             c for c in self._feature_names if c not in self.categorical_features
 12:         ]
 13: 
 14:         if fit:
 15:             self._scaler = StandardScaler()
 16:             if self._num_features:
 17:                 self._scaler.fit(X_proc[self._num_features])
 18: 
 19:             self._category_maps = {}
 20:             self._cat_cardinalities = []
 21:             for col in self.categorical_features:
 22:                 if col in X_proc.columns:
 23:                     unique_vals = sorted(X_proc[col].dropna().unique())


In [19]:
# -----------------------------
# Cell 15: Inspect categorical encoding behaviour
# -----------------------------

# Show only the part of _preprocess() that handles
# categorical mappings and transformed category values.

important_terms = [
    "category_maps",
    "cat_cardinalities",
    "unique_vals",
    "mapping",
    "map(",
    "fillna",
    "astype",
    "cat_arrays",
]

print("Relevant categorical preprocessing lines:\n")

for line_number, line in enumerate(
    preprocess_lines,
    start=1
):
    if any(
        term.lower() in line.lower()
        for term in important_terms
    ):
        print(f"{line_number:>3}: {line}")

Relevant categorical preprocessing lines:

 19:             self._category_maps = {}
 20:             self._cat_cardinalities = []
 23:                     unique_vals = sorted(X_proc[col].dropna().unique())
 24:                     mapping = {val: idx for idx, val in enumerate(unique_vals)}
 25:                     self._category_maps[col] = mapping
 26:                     self._cat_cardinalities.append(len(mapping))
 30:             x_num = self._scaler.transform(X_proc[self._num_features]).astype(np.float32)
 34:             cat_arrays = []
 37:                     mapping = self._category_maps.get(col, {})
 38:                     arr = X_proc[col].map(mapping).fillna(0).values.astype(np.int64)
 39:                     cat_arrays.append(arr)
 40:             if cat_arrays:
 41:                 x_cat = np.column_stack(cat_arrays)


In [20]:
# -----------------------------
# Cell 16: Check unseen categories across frozen CV folds
# -----------------------------

unseen_category_records = []


# For each outer validation fold:
# - use the other 4 folds as training data
# - learn which categorical values exist in training
# - check whether validation contains any new/unseen values
for fold in sorted(development_df["cv_fold"].unique()):

    train_fold_df = development_df[
        development_df["cv_fold"] != fold
    ]

    val_fold_df = development_df[
        development_df["cv_fold"] == fold
    ]

    for feature in categorical_feature_names:

        train_categories = set(
            train_fold_df[feature]
            .dropna()
            .unique()
        )

        val_categories = set(
            val_fold_df[feature]
            .dropna()
            .unique()
        )

        unseen_categories = (
            val_categories - train_categories
        )

        unseen_category_records.append(
            {
                "cv_fold": fold,
                "feature": feature,
                "train_category_count": len(train_categories),
                "validation_category_count": len(val_categories),
                "unseen_category_count": len(unseen_categories),
                "unseen_categories": sorted(unseen_categories),
            }
        )


unseen_category_summary = pd.DataFrame(
    unseen_category_records
)


display(unseen_category_summary)


# Total number of unseen categorical values
# across all feature/fold combinations.
total_unseen = int(
    unseen_category_summary[
        "unseen_category_count"
    ].sum()
)


print(
    "\nTotal unseen validation categories:",
    total_unseen
)


if total_unseen == 0:
    print(
        "No unseen categorical values occur in any "
        "of the five frozen validation folds."
    )
else:
    print(
        "\nWARNING: Some validation folds contain "
        "categories absent from their training folds."
    )

    display(
        unseen_category_summary[
            unseen_category_summary[
                "unseen_category_count"
            ] > 0
        ]
    )

,cv_fold,feature,train_category_count,validation_category_count,unseen_category_count,unseen_categories
0,1,Department,9,9,0,[]
1,1,Gender,3,3,0,[]
2,1,Job_Title,7,7,0,[]
3,1,Education_Level,4,4,0,[]
4,2,Department,9,9,0,[]
5,2,Gender,3,3,0,[]
6,2,Job_Title,7,7,0,[]
7,2,Education_Level,4,4,0,[]
8,3,Department,9,9,0,[]
9,3,Gender,3,3,0,[]



Total unseen validation categories: 0
No unseen categorical values occur in any of the five frozen validation folds.


## Verified FT-Transformer Training Protocol

Before training the model, the repository implementation was inspected to confirm the actual behavior of the FT-Transformer pipeline.

### Cross-validation

The shared `run_5fold_cv()` runner uses the frozen development folds:

- Fold 1
- Fold 2
- Fold 3
- Fold 4
- Fold 5

For each outer cross-validation round:

- **64,000 employees** are used for training.
- **16,000 employees** are used for validation.
- A fresh `FTTransformerAdapter` is created for every fold.
- The outer validation fold is never passed to `fit()`.
- One out-of-fold probability is produced for each development employee.

### Fold-local preprocessing

Inside each training fold:

- The 13 numerical/ordinal predictors are standardized using `StandardScaler`.
- The scaler is fitted using the training fold only.
- Categorical mappings are also created using the training fold only.
- Validation data is transformed using those already-fitted preprocessing objects.

All four categorical predictors were checked across all five frozen folds, and no validation fold contains a category that is absent from its corresponding training data.

### Actual neural-network training settings

The current repository implementation trains FT-Transformer using:

- **Optimizer:** AdamW
- **Loss:** Binary Cross-Entropy with Logits (`BCEWithLogitsLoss`)
- **Batch size:** 512
- **Training epochs:** 50
- **Learning rate:** 0.001 unless overridden through adapter parameters
- **Weight decay:** 1e-5 unless overridden through adapter parameters
- **Device:** CPU in the current environment

The configured FT-Transformer architecture uses:

- 3 Transformer blocks
- block dimension of 64
- 4 attention heads
- feed-forward hidden dimension of 128
- feed-forward dropout of 0.1
- attention dropout of 0.1
- residual dropout of 0.0
- random seed of 42

### Configuration note

The YAML configuration also contains:

- `max_epochs: 100`
- `patience: 15`
- `lr_scheduler: ReduceLROnPlateau`
- `validation_fraction_inner: 0.15`

However, inspection of the current adapter and shared CV runner showed that these validation settings are **not currently used by the implemented training loop**.

Therefore, results in this notebook will be reported according to the **training behavior actually executed by the repository code**, rather than claiming that unused YAML settings were applied.

### Evaluation priority

Model performance will be judged primarily using **PR-AUC**, with ROC-AUC, Brier Score, balanced accuracy, precision, recall, and F1 reported as supporting metrics.

In [21]:
# -----------------------------
# Cell 18: Create FT-Transformer adapter factory
# -----------------------------

def make_ft_transformer():
    """
    Return a fresh FT-Transformer adapter.

    The shared 5-fold CV runner will call this function
    once for each outer fold so that model state and
    preprocessing are never reused between folds.
    """

    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=fixed_params.copy(),
        categorical_features=categorical_feature_names.copy(),
    )


# --------------------------------------------------
# Verify that the factory creates independent models
# --------------------------------------------------

adapter_a = make_ft_transformer()
adapter_b = make_ft_transformer()


assert adapter_a is not adapter_b, (
    "The factory must create a new adapter instance each time."
)

assert adapter_a.params is not adapter_b.params, (
    "Parameter dictionaries should not be shared between adapters."
)

assert adapter_a.categorical_features is not adapter_b.categorical_features, (
    "Categorical feature lists should not be shared between adapters."
)

assert adapter_a.get_metadata()["fitted"] is False
assert adapter_b.get_metadata()["fitted"] is False


print("FT-Transformer factory created successfully.")

print(
    "Fresh adapter instances:",
    id(adapter_a) != id(adapter_b)
)

print(
    "Model:",
    adapter_a.name
)

print(
    "Model ID:",
    adapter_a.model_id
)

print(
    "Number of categorical features:",
    len(adapter_a.categorical_features)
)

print(
    "Both adapters initially unfitted:",
    (
        not adapter_a.get_metadata()["fitted"]
        and not adapter_b.get_metadata()["fitted"]
    )
)

FT-Transformer factory created successfully.
Fresh adapter instances: True
Model: FT-Transformer
Model ID: ft_transformer
Number of categorical features: 4
Both adapters initially unfitted: True


In [22]:
# -----------------------------
# Cell 19: Final FT-Transformer training preflight
# -----------------------------

from rtdl_revisiting_models import FTTransformer


# Confirm the external FT-Transformer implementation is available.
print(
    "FT-Transformer dependency:",
    FTTransformer.__module__
)

print(
    "PyTorch version:",
    torch.__version__
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Training device:",
    device
)


# --------------------------------------------------
# Check reproducibility logic inside adapter.fit()
# --------------------------------------------------

seed_terms = [
    "seed",
    "manual_seed",
    "cuda.manual_seed",
    "np.random",
    "random.seed",
]

seed_lines = []

for line_number, line in enumerate(
    fit_lines,
    start=1
):
    if any(
        term.lower() in line.lower()
        for term in seed_terms
    ):
        seed_lines.append(
            (line_number, line)
        )


print("\nReproducibility-related fit() lines:")

if seed_lines:
    for line_number, line in seed_lines:
        print(
            f"{line_number:>3}: {line}"
        )
else:
    print(
        "No explicit seed-setting lines were found "
        "inside FTTransformerAdapter.fit()."
    )


# --------------------------------------------------
# Final data/model checks
# --------------------------------------------------

assert X.shape == (80_000, 17)
assert len(cv_folds) == 80_000
assert set(cv_folds.unique()) == {1, 2, 3, 4, 5}
assert total_unseen == 0

print("\nFinal data checks passed.")
print("FT-Transformer is ready for CV training.")

FT-Transformer dependency: rtdl_revisiting_models
PyTorch version: 2.13.0+cpu
CUDA available: False
Training device: cpu

Reproducibility-related fit() lines:
 25:         # Set seed
 26:         torch.manual_seed(self.params.get("seed", 42))

Final data checks passed.
FT-Transformer is ready for CV training.


In [25]:
# -----------------------------
# Cell 20: Run 5-fold FT-Transformer cross-validation
# -----------------------------

import time


# --------------------------------------------------
# Build development-only raw modeling data
# --------------------------------------------------

# Get the Employee_ID values belonging to the development partition.
development_ids = set(
    split_registry.loc[
        split_registry["partition"] == "development",
        "Employee_ID"
    ]
)


# Keep only development employees from the original raw dataset.
#
# We deliberately use the raw dataframe here rather than development_df
# because run_5fold_cv() will align the frozen fold information from the
# split registry itself.
development_data = (
    df.loc[
        df["Employee_ID"].isin(development_ids)
    ]
    .copy()
    .reset_index(drop=True)
)


# --------------------------------------------------
# Leakage-safety checks
# --------------------------------------------------

holdout_ids = set(
    split_registry.loc[
        split_registry["partition"] == "holdout",
        "Employee_ID"
    ]
)


assert development_data.shape[0] == 80_000, (
    f"Expected 80,000 development rows, "
    f"found {len(development_data):,}."
)

assert development_data["Employee_ID"].is_unique, (
    "Employee_ID must be unique."
)

assert set(development_data["Employee_ID"]).isdisjoint(
    holdout_ids
), (
    "Formal holdout IDs were found in the data "
    "being passed to cross-validation."
)


print("Development-only modeling data:", development_data.shape)
print("Formal holdout IDs present:", 0)

print("\nStarting FT-Transformer 5-fold cross-validation...")
print("Training device: CPU")
print("Development rows: 80,000")
print("Predictors: 17")
print("Outer folds: 5")
print("Training rows per fold: 64,000")
print("Validation rows per fold: 16,000")
print("Epochs per fold: 50")
print("Batch size: 512")

print(
    "\nThis may take a while because five neural networks "
    "will be trained sequentially on CPU."
)


# --------------------------------------------------
# Run leakage-safe 5-fold CV
# --------------------------------------------------

cv_start_time = time.perf_counter()


ft_cv_result = run_5fold_cv(
    adapter_factory=make_ft_transformer,

    # IMPORTANT:
    # Only the 80,000 development employees enter the OOF runner.
    data=development_data,

    # Frozen split/fold assignments.
    split_registry=split_registry,

    # Exactly the 17 approved predictors.
    feature_cols=primary_model_features,

    target_col=target_column,
    id_col="Employee_ID",

    fold_col="cv_fold",
    partition_col="partition",

    development_label="development",
    holdout_label="holdout",

    expected_folds=(1, 2, 3, 4, 5),

    expected_model_name=model_name,
)


cv_elapsed_time = (
    time.perf_counter()
    - cv_start_time
)


print("\nFT-Transformer 5-fold CV completed.")

print(
    "Total CV training time:",
    f"{cv_elapsed_time / 60:.2f} minutes"
)

print(
    "Result object:",
    type(ft_cv_result).__name__
)

Development-only modeling data: (80000, 20)
Formal holdout IDs present: 0

Starting FT-Transformer 5-fold cross-validation...
Training device: CPU
Development rows: 80,000
Predictors: 17
Outer folds: 5
Training rows per fold: 64,000
Validation rows per fold: 16,000
Epochs per fold: 50
Batch size: 512

This may take a while because five neural networks will be trained sequentially on CPU.


TypeError: FTTransformerBackbone.__init__() missing 1 required keyword-only argument: 'ffn_d_hidden_multiplier'

In [26]:
# Check the exact FT-Transformer API provided
# by the currently installed rtdl_revisiting_models package.

import inspect
import rtdl_revisiting_models

from rtdl_revisiting_models import FTTransformer


print(
    "rtdl_revisiting_models version:",
    getattr(rtdl_revisiting_models, "__version__", "not exposed")
)

print("\nFTTransformer constructor:")
print(
    inspect.signature(FTTransformer)
)

print("\nFTTransformer.__init__:")
print(
    inspect.signature(FTTransformer.__init__)
)

rtdl_revisiting_models version: 0.0.2

FTTransformer constructor:
(*, n_cont_features: int, cat_cardinalities: List[int], _is_default: bool = False, **backbone_kwargs) -> None

FTTransformer.__init__:
(self, *, n_cont_features: int, cat_cardinalities: List[int], _is_default: bool = False, **backbone_kwargs) -> None


In [27]:
# -----------------------------
# Diagnostic: Inspect FTTransformerBackbone API
# -----------------------------

import inspect
import rtdl_revisiting_models


FTTransformerBackbone = (
    rtdl_revisiting_models.FTTransformerBackbone
)


print("FTTransformerBackbone constructor:")
print(
    inspect.signature(
        FTTransformerBackbone
    )
)

print("\nFTTransformerBackbone.__init__:")
print(
    inspect.signature(
        FTTransformerBackbone.__init__
    )
)

FTTransformerBackbone constructor:
(*, d_out: int | None, n_blocks: int, d_block: int, attention_n_heads: int, attention_dropout: float, ffn_d_hidden: int | None = None, ffn_d_hidden_multiplier: float | None, ffn_dropout: float, ffn_activation: Literal['ReLU', 'ReGLU'] = 'ReGLU', residual_dropout: float, n_tokens: int | None = None, linformer_kv_compression_ratio: float | None = None, linformer_kv_compression_sharing: Literal['headwise', 'key-value'] | None = None)

FTTransformerBackbone.__init__:
(self, *, d_out: int | None, n_blocks: int, d_block: int, attention_n_heads: int, attention_dropout: float, ffn_d_hidden: int | None = None, ffn_d_hidden_multiplier: float | None, ffn_dropout: float, ffn_activation: Literal['ReLU', 'ReGLU'] = 'ReGLU', residual_dropout: float, n_tokens: int | None = None, linformer_kv_compression_ratio: float | None = None, linformer_kv_compression_sharing: Literal['headwise', 'key-value'] | None = None)


In [28]:
# -----------------------------
# Diagnostic: Inspect model construction block
# -----------------------------

fit_lines = inspect.getsource(
    FTTransformerAdapter.fit
).splitlines()


# Show the section around:
# self._model = FTTransformer(...)
for line_number in range(25, 55):

    if line_number <= len(fit_lines):

        print(
            f"{line_number:>3}: "
            f"{fit_lines[line_number - 1]}"
        )

 25:         # Set seed
 26:         torch.manual_seed(self.params.get("seed", 42))
 27: 
 28:         n_cont_features = x_num.shape[1] if x_num is not None else 0
 29:         cat_cardinalities = self._cat_cardinalities if x_cat is not None else None
 30: 
 31:         self._model = FTTransformer(
 32:             n_cont_features=n_cont_features,
 33:             cat_cardinalities=cat_cardinalities,
 34:             d_out=1,
 35:             n_blocks=self.params.get("n_blocks", 3),
 36:             d_block=self.params.get("d_block", 64),
 37:             attention_n_heads=self.params.get("attention_n_heads", 4),
 38:             attention_dropout=self.params.get("attention_dropout", 0.1),
 39:             ffn_d_hidden=self.params.get("ffn_d_hidden", 128),
 40:             ffn_dropout=self.params.get("ffn_dropout", 0.1),
 41:             residual_dropout=self.params.get("residual_dropout", 0.0),
 42:         ).to(device)
 43: 
 44:         tensors = []
 45:         if x_num is not None

In [29]:
# -----------------------------
# Cell 20A: Reload adapter and run a small compatibility test
# -----------------------------

import importlib
import numpy as np

import src.models.ft_transformer as ft_module


# Reload the module so the notebook uses the file we just updated.
ft_module = importlib.reload(ft_module)

# Replace the old imported class with the updated one.
FTTransformerAdapter = ft_module.FTTransformerAdapter


# --------------------------------------------------
# Create a small development-only sample
# --------------------------------------------------

smoke_df = (
    development_df
    .sample(
        n=2_048,
        random_state=42
    )
    .reset_index(drop=True)
)

X_smoke = smoke_df[
    primary_model_features
].copy()

y_smoke = smoke_df[
    target_column
].astype(int).copy()


print("Smoke-test rows:", len(X_smoke))
print("Positive cases:", int(y_smoke.sum()))


# --------------------------------------------------
# Create the updated adapter
# --------------------------------------------------

smoke_adapter = FTTransformerAdapter(
    name=model_name,
    model_id=model_id,
    params=fixed_params.copy(),
    categorical_features=categorical_feature_names.copy(),
)


# Train for ONLY 1 epoch.
#
# This is not part of the final model evaluation.
# It only checks that model construction, forward pass,
# backward pass, and prediction all work correctly.
smoke_adapter.fit(
    X=X_smoke,
    y=y_smoke,
    epochs=1,
    batch_size=256,
)


# Generate a few probabilities.
smoke_probabilities = smoke_adapter.predict_proba(
    X_smoke.iloc[:64]
)


# --------------------------------------------------
# Basic validation
# --------------------------------------------------

assert smoke_adapter.get_metadata()["fitted"] is True

assert len(smoke_probabilities) == 64

assert np.isfinite(smoke_probabilities).all()

assert (
    (smoke_probabilities >= 0)
    & (smoke_probabilities <= 1)
).all()


print("\nCompatibility smoke test PASSED.")

print(
    "Prediction count:",
    len(smoke_probabilities)
)

print(
    "Probability range:",
    f"{smoke_probabilities.min():.6f}",
    "to",
    f"{smoke_probabilities.max():.6f}"
)

Smoke-test rows: 2048
Positive cases: 194

Compatibility smoke test PASSED.
Prediction count: 64
Probability range: 0.077371 to 0.085327


In [30]:
# -----------------------------
# Cell 20B: Official 5-fold FT-Transformer cross-validation
# -----------------------------

import time


# --------------------------------------------------
# Rebuild the factory using the reloaded adapter class
# --------------------------------------------------

def make_ft_transformer():
    """
    Create a completely fresh FT-Transformer adapter
    for each outer CV fold.
    """

    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=fixed_params.copy(),
        categorical_features=categorical_feature_names.copy(),
    )


# Quick confirmation that the updated adapter is being used.
factory_check = make_ft_transformer()

assert isinstance(
    factory_check,
    FTTransformerAdapter
)

assert factory_check.get_metadata()["fitted"] is False


# --------------------------------------------------
# Final holdout-isolation check
# --------------------------------------------------

development_ids = set(
    split_registry.loc[
        split_registry["partition"] == "development",
        "Employee_ID"
    ]
)

holdout_ids = set(
    split_registry.loc[
        split_registry["partition"] == "holdout",
        "Employee_ID"
    ]
)


development_data = (
    df.loc[
        df["Employee_ID"].isin(development_ids)
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(development_data) == 80_000

assert set(
    development_data["Employee_ID"]
).isdisjoint(
    holdout_ids
), "Holdout leakage detected."


# --------------------------------------------------
# Training summary
# --------------------------------------------------

print("FT-Transformer official CV training")
print("-----------------------------------")
print("Development rows:", len(development_data))
print("Formal holdout rows used: 0")
print("Predictors:", len(primary_model_features))
print("Outer CV folds:", 5)
print("Training rows per fold:", 64_000)
print("Validation rows per fold:", 16_000)
print("Epochs per fold:", 50)
print("Batch size:", 512)
print("Device:", "cpu")

print(
    "\nStarting training..."
)

print(
    "Because training is running on CPU, "
    "this cell may take a substantial amount of time."
)


# --------------------------------------------------
# Run canonical leakage-safe 5-fold CV
# --------------------------------------------------

cv_start_time = time.perf_counter()


ft_cv_result = run_5fold_cv(
    adapter_factory=make_ft_transformer,

    # ONLY development employees are supplied.
    data=development_data,

    # Frozen split assignments from Notebook 02.
    split_registry=split_registry,

    # Exactly the approved 17 predictors.
    feature_cols=primary_model_features,

    target_col=target_column,
    id_col="Employee_ID",

    fold_col="cv_fold",
    partition_col="partition",

    development_label="development",
    holdout_label="holdout",

    expected_folds=(1, 2, 3, 4, 5),

    expected_model_name=model_name,
)


cv_elapsed_time = (
    time.perf_counter()
    - cv_start_time
)


# --------------------------------------------------
# Completion summary
# --------------------------------------------------

print(
    "\nFT-Transformer 5-fold CV completed successfully."
)

print(
    "Total elapsed time:",
    f"{cv_elapsed_time / 60:.2f} minutes"
)

print(
    "Returned result type:",
    type(ft_cv_result).__name__
)

FT-Transformer official CV training
-----------------------------------
Development rows: 80000
Formal holdout rows used: 0
Predictors: 17
Outer CV folds: 5
Training rows per fold: 64000
Validation rows per fold: 16000
Epochs per fold: 50
Batch size: 512
Device: cpu

Starting training...
Because training is running on CPU, this cell may take a substantial amount of time.


KeyboardInterrupt: 